# RNA+ADT mapping and prediction

This tutorial trains CONNECT on paired RNA+ADT data and exports both latent-space mappings and cross-modal predictions.


## 1. Load data


In [ ]:
import os
from os.path import join

import scanpy as sc

from connect import (
    build_model,
    build_paired_loader,
    get_device,
    init_logger,
    make_modality,
    predict,
    save_model,
    save_outputs,
    set_seed,
    train_model,
)

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


In [ ]:
data_dir = "/path/to/rna_adt_dataset"
save_dir = "./connect_rna_adt_result"

set_seed(42)
device = get_device("0")
logger = init_logger(save_dir)

rna_train = sc.read_h5ad(join(data_dir, "rna_train.h5ad"))
adt_train = sc.read_h5ad(join(data_dir, "adt_train.h5ad"))
rna_test = sc.read_h5ad(join(data_dir, "rna_test.h5ad"))
adt_test = sc.read_h5ad(join(data_dir, "adt_test.h5ad"))


## 2. Describe preprocessing

ADT uses CLR normalization through `preprocess=True`.


In [ ]:
train_rna = make_modality(rna_train, "RNA", preprocess=False)
train_adt = make_modality(adt_train, "ADT", preprocess=True)
test_rna = make_modality(rna_test, "RNA", preprocess=False)
test_adt = make_modality(adt_test, "ADT", preprocess=True)


## 3. Build loaders and model


In [ ]:
data_loader, valid_data_loader = build_paired_loader(
    train_rna,
    train_adt,
    batch_size=128,
    validation_split=0.1,
    num_workers=2,
    training=True,
    logger=logger,
)

model = build_model(data_loader, train_rna, train_adt, device=device)
logger.info(model)


## 4. Standard paired training


In [ ]:
model = train_model(
    model,
    data_loader,
    valid_data_loader,
    device=device,
    epochs=80,
    lr=1e-3,
    weight_decay=1e-3,
    amsgrad=True,
    logger=logger,
)


## 5. Infer mapping and prediction outputs


In [ ]:
outputs = predict(
    model,
    test_rna,
    test_adt,
    device=device,
    batch_size=128,
    num_workers=2,
    logger=logger,
)

rna_test.obsm["connect_rna_emb"] = outputs["mod1_latent"].cpu().numpy()
rna_test.obsm["connect_rna_mapped_from_adt"] = outputs["mod1_mapping_from_mod2"].cpu().numpy()
rna_test.layers["connect_rna_predicted_from_adt"] = outputs["mod1_predicted_from_mod2"].cpu().numpy()

adt_test.obsm["connect_adt_emb"] = outputs["mod2_latent"].cpu().numpy()
adt_test.obsm["connect_adt_mapped_from_rna"] = outputs["mod2_mapping_from_mod1"].cpu().numpy()
adt_test.layers["connect_adt_predicted_from_rna"] = outputs["mod2_predicted_from_mod1"].cpu().numpy()

save_model(model, save_dir)
save_outputs(outputs, save_dir, filename="rna_adt_outputs.pt")
